# 3. データを下ごしらえする

前の章で、データに2つの問題が見つかった。

- `age` と `embarked` に**値が入っていない人がいる**
- `sex` と `embarked` が**文字**で入っている

この章で両方を片付けて、**すべて数字で、空いている場所がない**状態にする。
料理の下ごしらえと同じで、ここを済ませておかないと次の工程に進めない。

## 1. 学習用と評価用をいったん繋げる

下ごしらえは、学習用と評価用の**両方に同じ処理**をしなければならない。
学習用だけ整えても、評価用が空いたままでは予測できない。

同じコードを2回書くと、片方だけ直し忘れる事故が起きる。
そこで一度たてに繋げて1つの表にして処理し、最後に分け直す。

In [ ]:
import pandas as pd

train = pd.read_csv("../../data/raw/train.csv", index_col=0)
test = pd.read_csv("../../data/raw/test.csv", index_col=0)

# 後で分け直すために、どの id が train でどの id が test だったかを控えておく
#   結合すると両者の区別が付かなくなるため
train_index = train.index
test_index = test.index

In [ ]:
# pd.concat() は表を繋げる。何も指定しなければ、たてに積み重ねる
data = pd.concat([train, test])

print("train:", train.shape, "+ test:", test.shape, "→ data:", data.shape)
print()
print(data.dtypes)

# 出力の見方
#   445 人 + 446 人 = 891 人。項目は train に合わせて 8 個になる
#   test には survived が無いので、test 側の 446 人分は「空」になる
#   その結果 survived が整数から小数に変わる。
#   空きがある項目を pandas が小数として扱うため（01 の age と同じ理由）

### 出力の最後に付く `dtype: object` とは

上の出力の最後に `dtype: object` という行が付いている。これは**データではない**。

`data.dtypes` が返しているのは「項目名」と「その項目の種類」が対になった一覧で、
pandas ではこういう1列だけの表を **Series** と呼ぶ。
Series を表示すると、最後に必ず「この表は何を入れているか」が添えられる。

```
survived    float64     ← 左が項目名、右がその項目の種類
pclass        int64
sex             str
...
dtype: object           ← この一覧自体が何を入れているか（データではない）
```

`object` は「数字でも文字でもない、その他いろいろ」という意味。
ここに入っているのは `float64` のような**種類を表すもの**なので、その他になる。

この行は Series を表示すれば毎回出てくる。

| 出力 | 最後の行 | なぜ |
| --- | --- | --- |
| `value_counts()` | `Name: count, dtype: int64` | 人数を数えたので整数 |
| `corrwith()` | `dtype: float64` | 相関係数は小数 |
| `dtypes` | `dtype: object` | 中身が「種類」なのでその他 |

In [ ]:
# 繋げた後、値が空いている場所を数える
data.isna().sum()

# 出力の見方
#   survived  446 → test 側。これは埋める対象ではなく、これから予測するもの
#   age       177 → train 85 人 + test 92 人。これが埋める対象
#   embarked    2 → train のみ。これも埋める対象

### `isna()` の読み方

`isna` は **is NA**、つまり「NA ですか？」と尋ねる命令。
**NA** は **Not Available** の略で、「値が手に入っていない」という意味。

pandas で「値がない」を表す印は1種類ではないが、`isna()` はまとめて `True` にしてくれる。

| 画面での表示 | 正体 |
| --- | --- |
| `NaN` | Not a Number の略。数字が入るはずの場所が空のときの印 |
| `None` | Python がもともと持っている「何もない」 |
| `<NA>` | pandas が用意した空っぽ専用の印 |

CSV の空欄は、数字の項目なら `NaN` として読み込まれる。

`isna()` が返すのは、元と**同じ形の「はい／いいえ」の表**。

```
      age  embarked
id
3   False     False    ← どちらも値が入っている
4   False     False
```

ここに `.sum()` を付けると、`True` を 1 として足すので、空いている個数になる。
`.sum().sum()` まで付ければ、表全体の合計が1つの数字で出る。

逆を知りたいときは `notna()`（値が入っていれば `True`）を使う。

## 2. 空いているところを埋める

埋め方に唯一の正解はない。やり方は大きく3つある。

| やり方 | 向いている場面 | 今回 |
| --- | --- | --- |
| その人の行をまるごと捨てる | 空きがごく少数のとき | `age` は 891 人中 177 人（約2割）が空き。捨てると学習データが大幅に減るので**使わない** |
| その項目をまるごと捨てる | ほとんど空きのとき | `age` は8割は埋まっているので、捨てるのは惜しい |
| **代表的な値で埋める** | 上記以外 | **これを選ぶ** |

何で埋めるかは、項目の種類で決まる。

- `age`（数字）→ **平均値**
- `embarked`（港の名前）→ **一番多い値**

港に平均は存在しないので、文字の項目に平均は使えない。

In [ ]:
# .fillna(値) は、空いているところをその値で埋める
#   .mean() は平均値
data["age"] = data["age"].fillna(data["age"].mean())

print("埋めた値（全体の平均年齢）:", round(data["age"].mean(), 2))
print("age の空き:", data["age"].isna().sum())

# 平均で埋めるか真ん中の値で埋めるかは悩みどころ。
# 今回の age は 平均 29.70 / 真ん中 28.0 とほぼ同じなので、どちらでも結果は変わらない。
# 運賃のように極端な値が混ざる項目なら、真ん中の値の方が安全

In [ ]:
# .mode() は一番多く出てくる値を求める
#   末尾の [0] が大事。理由は下で説明する
data["embarked"] = data["embarked"].fillna(data["embarked"].mode()[0])

print("埋めた値（一番多い港）:", data["embarked"].mode()[0])
print("embarked の空き:", data["embarked"].isna().sum())

### `.mode()` に `[0]` を付ける理由

`.mode()` が返すのは値そのものではなく、**値の入ったリスト**。
一番多い値が同じ数で並ぶこともあるので、1つしかなくても常にリストで返ってくる。
`[0]` は、そのリストの1番目を取り出す指定。

付け忘れると、`.fillna()` が「この値で埋めろ」ではなく
「番号を突き合わせて埋めろ」と受け取ってしまい、
**エラーも出ないまま1つも埋まらない**。
埋めた直後に空きの数を表示しているのは、これに気づくため。

### 同じ数で並んだらどうするのか

`[0]` は「アルファベット順で最初のもの」を選ぶ。適当に見えるが、
**何回実行しても同じ結果になる**のが利点。

ランダムに選ぶ手もあるが使わない。下ごしらえにサイコロが入ると、
実行するたびにデータが変わってしまう。すると後で精度が上がったとき、
工夫が効いたのか、たまたま運が良かったのかを区別できなくなる。

なお今回の `embarked` は `S` が 644 人、`C` が 168 人、`Q` が 77 人で、並んでいない。
`[0]` が必要なのは、**返ってくる形がリストで決まっている**からにすぎない。

## 3. 文字を数字に置き換える

文字のままでは次の章の仕組みに渡せないので、02 で使った `get_dummies()` で
0/1 の項目に置き換える。

In [ ]:
data = pd.get_dummies(data)

print(data.shape)
print()
print(data.dtypes)

# 出力の見方
#   項目が 8 個 → 11 個に増えた
#     sex（1 個）      → sex_female, sex_male（2 個）
#     embarked（1 個） → embarked_C, embarked_Q, embarked_S（3 個）
#   文字の項目が無くなり、数字と True/False だけになった

## 4. 学習用と評価用に分け直す

最初に控えておいた id を使って、元の2つに戻す。

In [ ]:
# .loc[id のリスト] で、指定した id の行だけを取り出す
train = data.loc[train_index]
test = data.loc[test_index]

# test の survived は結合時に付いた空の列なので落とす
#   columns= を使うと「列を落とす」ことが名前から分かる
#   行を落とすときは index= を使う
test = test.drop(columns=["survived"])

print("train:", train.shape)  # 11 列（survived を含む）
print("test:", test.shape)  # 10 列（survived を除いた分だけ少ない）

In [ ]:
train.head()

# 出力の見方
#   survived が 1.0 / 0.0 と小数なのは、結合したときに float64 になった名残
#   train 側は全員分の答えが揃っているので、欠損はない

In [ ]:
# 仕上げの確認。ここが全て 0 でなければ、モデルに渡す前に原因を潰す
print("train の欠損:", train.isna().sum().sum())
print("test の欠損:", test.isna().sum().sum())
print("文字列の列:", list(train.select_dtypes("str").columns))

## 5. 次の章で使えるように保存する

Notebook は1つずつが独立して動くので、ここで作った `train` と `test` は
この Notebook を閉じると消えてしまう。ファイルに書き出しておく。

保存先の `data/processed/` は git に記録されない設定になっている。
競技のデータは配り直してはいけない決まりなので、加工した後のものも同じ扱いにしている。

In [ ]:
# index=True で id も一緒に書き出す。id を失うと提出時にどの乗客の予測か分からなくなる
train.to_csv("../../data/processed/train_processed.csv", index=True)
test.to_csv("../../data/processed/test_processed.csv", index=True)

print("保存した")

## この章のまとめ

- 学習用と評価用を**繋げてから**処理した。同じ作業を確実に両方へ届けるため
- `age` は平均値（29.70）、`embarked` は一番多い値（`S`）で埋めた
- `.mode()` はリストを返すので `[0]` が必要。忘れると**エラーも出ずに埋まらない**
- `get_dummies()` で文字を 0/1 に置き換え、項目が 8 個 → 11 個になった
- 結果を `data/processed/` の2つのファイルに保存した

ひとつ気をつけたい点がある。今回は学習用と評価用を繋げた状態で平均値を求めている。
手順は簡単になるが、**本当は知らないはずの評価用データの様子を使って**
学習用データを加工していることになる。

学習用だけから求めた値で両方を埋める方が正しいので、
精度を上げていく段階で見直す余地がある。

次はいよいよ、予測する仕組みを作る。